# LAST UPDATE 18/04/2025

In [20]:
import pandas as pd
import os

### TASK 1: IMPORT AND MERGE 12-MONTH DATA

In [23]:
##Create a path
path = "C:\\Users\\MY PC\\Downloads\\Casestudy_How_does_a_bike-share_navigate_speedy_success\\Customer_Data\\"

In [27]:
# 1: Lấy danh sách đường dẫn vào từng tập tin
frames = [] 
all_length = []
for file in os.listdir(path): # Chạy từng tập tin trong danh sách này
    filepath = path + file # tạo path cho từng tập tin
    df1 = pd.read_csv(filepath) # tạo dataframe cho từng tập tin dựa trện path của nó
    frames.append(df1) # thêm data frame đó vào danh sách frames[]
    result = pd.concat(frames) # truyền danh sách vào phương thức concat 
    length_1month = len(df1.index)
    all_length.append(length_1month)
    
df = result
df

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,4422E707103AA4FF,electric_bike,10/14/2024 3:26,10/14/2024 3:32,NaN,NaN,NaN,NaN,41.960000,-87.650000,41.98,-87.67,member
1,19DB722B44CBE82F,electric_bike,10/13/2024 19:33,10/13/2024 19:39,NaN,NaN,NaN,NaN,41.980000,-87.670000,41.97,-87.66,member
2,20AE2509FD68C939,electric_bike,10/13/2024 23:40,10/13/2024 23:48,NaN,NaN,NaN,NaN,41.970000,-87.660000,41.95,-87.65,member
3,D0F17580AB9515A9,electric_bike,10/14/2024 2:13,10/14/2024 2:25,NaN,NaN,NaN,NaN,41.950000,-87.650000,41.96,-87.65,member
4,A114A483941288D1,electric_bike,10/13/2024 19:26,10/13/2024 19:28,NaN,NaN,NaN,NaN,41.980000,-87.670000,41.98,-87.67,member
...,...,...,...,...,...,...,...,...,...,...,...,...,...
821271,97CC940225245B6C,electric_scooter,9/26/2024 12:09,9/26/2024 12:18,Franklin St & Adams St (Temp),TA1309000008,NaN,NaN,41.879339,-87.635700,41.90,-87.64,member
821272,4F8F2383056480E0,electric_bike,9/10/2024 18:10,9/10/2024 18:21,Streeter Dr & Grand Ave,13022,NaN,NaN,41.892278,-87.612043,41.86,-87.61,member
821273,6A60FA5CF71D36E6,electric_bike,9/14/2024 10:52,9/14/2024 11:03,Halsted St & Clybourn Ave,331,NaN,NaN,41.909701,-87.648408,41.90,-87.63,member
821274,F257ABC8B92ED5E5,electric_bike,9/30/2024 10:52,9/30/2024 11:31,Streeter Dr & Grand Ave,13022,NaN,NaN,41.892278,-87.612043,41.88,-87.62,member


## Check if the number of rows in df matches the total number of rows of 12 months?

In [29]:
print(sum(all_length))

5860568


## Save the 12-month file

In [32]:
df.to_csv('customer_data_2024.csv',index = False)

### TASK 2: CLEANING DATA

In [34]:
##remove rows contain missing value
df = df.dropna()

In [35]:
##remove rows contain duplicates value
df = df.drop_duplicates()

In [36]:
#Remove data with greater start_at than end_at
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])
df = df[df['started_at'] <= df['ended_at']]

In [37]:
#format timeline in dataframes for slice month, day, time
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])
df['started_at'] = df['started_at'].astype(str).replace('-', '/', regex=True)
df['ended_at'] = df['ended_at'].astype(str).replace('-', '/', regex=True)

In [38]:
#slice day,month, weekday
df['month'] = df['started_at'].str[5:7]
df['day'] = df['started_at'].str[8:10]
#change datatype of started_at for finding day of week
df['started_at'] = pd.to_datetime(df['started_at'])
df['day_of_week'] = df['started_at'].dt.day_name()
#slice time
df['time'] = df['started_at'].dt.strftime('%H:%M')
#add ride_length collumn
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])
df['ride_length'] = df['ended_at'] - df['started_at']
df['ride_length'] = df['ride_length'].apply(lambda x: x.total_seconds() / 60)
df['ride_length'] = df['ride_length'].astype(int)

In [39]:
#remove stolen bikes
df = df[df['ride_length'] <= 1440]
df = df[df['ride_length'] >= 5]

In [40]:
#keep the columns we will use
columns_to_keep = ['rideable_type', 'member_casual','month','time','day','day_of_week', 'ride_length']
df = df[columns_to_keep]

### TASK 3: SAVE THE DATA FOR ANALYSIS

In [42]:
#save the data for analysis
df.to_csv('cleaned_data_2024.csv',index = False)

In [43]:
df.head(10)

,rideable_type,member_casual,month,time,day,day_of_week,ride_length
370,classic_bike,member,10,19:20,01,Tuesday,15
1438,classic_bike,casual,10,18:59,02,Wednesday,7
1547,classic_bike,casual,10,18:59,02,Wednesday,8
1605,electric_bike,member,10,16:01,05,Saturday,12
2148,electric_bike,member,10,12:18,27,Sunday,7
2172,electric_bike,member,10,10:41,22,Tuesday,20
2353,classic_bike,casual,10,15:05,20,Sunday,11
2772,electric_bike,casual,10,23:29,05,Saturday,35
2890,classic_bike,casual,10,19:51,03,Thursday,36
3712,classic_bike,member,10,04:18,04,Friday,48
